<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/notebooks/02-visualizacao_resultados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Célula 1: Configuração e Criação da Pasta de Exportação
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from google.colab import drive
import os

# 1. Montagem do Drive
drive.mount('/content/drive')

# 2. Definição de Caminhos (AJUSTE O CAMINHO DO SEU BANCO ABAIXO)
DB_PATH = '/content/drive/My Drive/mba-engsof-tcc/v5/base-dados-v5.db'
EXPORT_PATH = '/content/drive/My Drive/mba-engsof-tcc/v5/graficos-tcc'

# Cria a pasta de exportação se ela não existir
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)
    print(f"📂 Pasta criada: {EXPORT_PATH}")

def get_connection():
    return sqlite3.connect(DB_PATH)

# Garante suporte a acentuação e visual limpo
sns.set_context("paper", font_scale=1.2)

# Configurações para qualidade ds imagens
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

print("✅ Ambiente configurado para exportação de imagens JPG.")

In [ ]:
# Célula 2: Geração de Nuvens de Palavras (Apenas Versos Positivos)
import nltk
from nltk.corpus import stopwords
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import os

nltk.download('stopwords')

def executar_nuvens_antidotos_positivos():
    conn = get_connection()

    # 1. Seleciona os eixos (excluindo o narrativo)
    query_antidotos = "SELECT DISTINCT antidoto_referencia FROM topico WHERE id != 3"
    lista_antidotos = pd.read_sql_query(query_antidotos, conn)['antidoto_referencia'].tolist()

    # 2. Stopwords focadas em limpar resíduos estruturais
    stop_words_pt = set(stopwords.words('portuguese'))
    custom_stops = {
        'disse', 'então', 'veio', 'porque', 'pois', 'sobre', 'todos', 'tudo',
        'assim', 'ainda', 'outra', 'outros', 'será', 'pode', 'fazer', 'tão',
        'casa', 'filho', 'filhos', 'homem', 'mulher', 'terra', 'povo', 'rei',
        'senhor', 'deus', 'jesus', 'cristo', 'amém', 'ora', 'eis', 'vós',
        'teu', 'tua', 'meu', 'minha', 'toda', 'ano', 'anos', 'morreu', 'mortos'
    }
    todas_stops = stop_words_pt.union(custom_stops)

    print("🌟 Gerando Nuvens de Antídotos (Filtro: Sentimento Positivo)...")

    for nome_antidoto in lista_antidotos:
        # CONSULTA MODIFICADA: Adicionamos o filtro s.sentimento_num = 1
        query_texto = f"""
            SELECT vl.texto_limpo
            FROM verso_limpo vl
            JOIN verso_topico vt ON vl.verso_id = vt.verso_id
            JOIN topico t ON vt.topico_id = t.id
            JOIN verso_sentimento s ON vl.verso_id = s.verso_id
            WHERE t.antidoto_referencia = '{nome_antidoto}'
            AND s.sentimento_num = 1
        """
        df_textos = pd.read_sql_query(query_texto, conn)

        if not df_textos.empty:
            texto_final = " ".join(df_textos['texto_limpo'].fillna('').tolist())

            # Estética mais sóbria para o TCC
            wordcloud = WordCloud(width=1600, height=1000,
                                  background_color='white',
                                  max_words=70,
                                  stopwords=todas_stops,
                                  colormap='plasma',
                                  collocations=False,
                                  prefer_horizontal=0.9).generate(texto_final)

            plt.figure(figsize=(12, 8))
            plt.imshow(wordcloud, interpolation='bilinear')
            plt.axis('off')

            safe_name = nome_antidoto.split('(')[0].strip().lower().replace(' ', '_')
            file_name = f"nuvem_positiva_{safe_name}.jpg"

            plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight')
            plt.close()
            print(f"✅ Nuvem de 'Cura' salva: {file_name}")
        else:
            print(f"⚠️ Sem versos positivos suficientes para: {nome_antidoto}")

    conn.close()

executar_nuvens_antidotos_positivos()

In [ ]:
# Célula: Distribuição Temática por Gênero Literário (Minimalista com Eixos)
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def gerar_grafico_distribuicao_tematica_minimalista():
    conn = get_connection()

    # 1. Consulta SQL para agrupar os 9 tópicos por Antídoto e Gênero
    query = """
        SELECT
            g.nome as Genero,
            t.antidoto_referencia as Antidoto,
            COUNT(vt.verso_id) as Frequencia
        FROM verso_topico vt
        JOIN topico t ON vt.topico_id = t.id
        JOIN verso v ON vt.verso_id = v.id
        JOIN livro l ON v.livro_id = l.id
        JOIN genero_literario g ON l.genero_id = g.id
        WHERE t.antidoto_referencia IN (
            'Antídoto à Insignificância (Sentido/Esperança)',
            'Antídoto ao Esgotamento (Vigor/Descanso)',
            'Antídoto à Transitoriedade (Eternidade/Rocha)'
        )
        GROUP BY Genero, Antidoto
    """

    df_dist = pd.read_sql_query(query, conn)
    conn.close()

    if df_dist.empty:
        print("⚠️ Dados não encontrados.")
        return

    # 2. Pivotar os dados para o formato de pilhas
    df_pivot = df_dist.pivot(index='Genero', columns='Antidoto', values='Frequencia').fillna(0)
    # Ordenação ascendente para que os maiores fiquem no topo do gráfico de barras horizontais
    df_pivot = df_pivot.sort_values(by=df_pivot.columns.tolist(), ascending=True)

    # 3. Configuração do Gráfico
    sns.set_style("white") # Fundo branco sólido sem grades

    # Paleta de cores solicitada/definida
    cores = ['#4A90E2', '#50C878', '#F5A623']

    ax = df_pivot.plot(kind='barh', stacked=True, color=cores, figsize=(14, 8), width=0.8)

    # Títulos e Rótulos (Título removido, Eixos mantidos)
    plt.title('')
    plt.xlabel('Quantidade de Versículos Associados', fontsize=12)
    plt.ylabel('Gênero Literário', fontsize=12)

    # Removendo apenas as bordas (moldura), mantendo a integridade dos eixos
    sns.despine(left=False, bottom=False, trim=False)

    # Ajustando a legenda (sem borda)
    plt.legend(title='Categorias de Antídoto', bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)

    plt.tight_layout()

    # 4. Salvamento no Drive
    file_name = "distribuicao_topicos_minimalista_eixos.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, format='jpg', bbox_inches='tight')

    print(f"✅ Gráfico minimalista com eixos salvo em: {file_name}")
    plt.show()

# Executa a geração do gráfico
gerar_grafico_distribuicao_tematica_minimalista()

In [ ]:
# Célula: Gauge Charts Minimalistas (Apenas Arco e Escore)
import matplotlib.pyplot as plt
import numpy as np
import os

def criar_gauge_ultra_minimal(valor, nome_arquivo):
    # Limite da escala fixo em 0.5 para garantir comparabilidade entre os gráficos
    limite_escala = 0.5

    # Cálculo proporcional para o semicírculo (180 graus)
    fatia_valor = (valor / limite_escala) * 180
    fatia_restante = 180 - fatia_valor

    fig, ax = plt.subplots(figsize=(6, 3))

    # Cores: Verde (Valor), Cinza Ultra-Claro (Fundo), Branco (Ocultar base)
    cores = ['#50C878', '#F2F2F2', 'white']

    # Estrutura do gráfico: [Valor, Fundo do Arco, Base invisível]
    ax.pie([fatia_valor, fatia_restante, 180],
           colors=cores,
           startangle=180,
           counterclock=True,
           wedgeprops={'width': 0.4, 'edgecolor': 'white'})

    # Escore numérico centralizado e em destaque
    plt.text(0, 0.05, f"{valor:.2f}", horizontalalignment='center',
             verticalalignment='center', fontsize=26, fontweight='bold', color='#333333')

    # Remoção total de eixos, títulos e textos secundários
    ax.axis('equal')
    plt.title('')

    # Salvamento no Drive com DPI alto para impressão
    path_completo = os.path.join(EXPORT_PATH, nome_arquivo)
    plt.savefig(path_completo, dpi=300, format='jpg', bbox_inches='tight', pad_inches=0.05)

    plt.show()
    print(f"💾 Gauge salvo: {nome_arquivo}")

# --- Execução direta dos medidores puros ---
print("🚀 Gerando medidores ultra-minimalistas...")

criar_gauge_ultra_minimal(0.19, "gauge_esgotamento_puro.jpg")
criar_gauge_ultra_minimal(0.08, "gauge_transitoriedade_puro.jpg")
criar_gauge_ultra_minimal(0.06, "gauge_insignificancia_puro.jpg")

In [ ]:
# Célula: Amplitude Emocional por Gênero (Mín, Máx e Média)
import pandas as pd
import matplotlib.pyplot as plt
import os

def gerar_grafico_amplitude_emocional():
    # Dados reais da sua consulta
    data = {
        'Genero': ['Apocalíptico', 'Epístola', 'Evangelho', 'Histórico', 'Pentateuco', 'Poético/Sapiencial', 'Profético'],
        'Min': [-0.978, -0.985, -0.989, -0.980, -0.990, -0.989, -0.991],
        'Max': [0.955, 0.979, 0.960, 0.957, 0.926, 0.991, 0.957],
        'Med': [0.144, 0.172, -0.163, 0.048, -0.209, 0.169, -0.393]
    }
    df = pd.DataFrame(data).sort_values(by='Med')

    plt.figure(figsize=(12, 7))

    # 1. Desenha a amplitude (do Min ao Max) como uma linha cinza sutil
    for i, row in df.iterrows():
        plt.plot([row['Min'], row['Max']], [row['Genero'], row['Genero']],
                 color='#D1D1D1', linewidth=4, alpha=0.6, zorder=1)

    # 2. Desenha a Média como um ponto colorido (Divergente)
    # Vermelho para médias negativas, Azul para médias positivas
    cores = ['#D9534F' if x < 0 else '#4A90E2' for x in df['Med']]
    plt.scatter(df['Med'], df['Genero'], color=cores, s=120, zorder=2, edgecolors='white')

    # 3. Linha vertical no Zero (Equilíbrio)
    plt.axvline(0, color='#333333', linestyle='--', linewidth=0.8, alpha=0.5)

    # Ajustes Minimalistas
    plt.xlabel('Amplitude de Sentimento (Mínimo, Máximo e Média)', fontsize=11)
    plt.ylabel('')
    plt.xlim(-1.1, 1.1) # Escala completa do BERTimbau

    sns.despine(left=True, bottom=False)
    plt.grid(False)
    plt.tight_layout()

    # Salvamento
    file_name = "amplitude_emocional_genero.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, format='jpg', bbox_inches='tight')

    print(f"✅ Gráfico de amplitude salvo: {file_name}")
    plt.show()

gerar_grafico_amplitude_emocional()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os

def gerar_workflow_data_prep_destaque():
    # Proporção ampla para garantir que as fontes maximizadas respirem
    fig, ax = plt.subplots(figsize=(24, 7), facecolor='white')
    ax.set_facecolor('white')

    etapas = [
        {"n": "1", "titulo": "Ingestão", "sub": "Business Und.", "libs": "google.colab\nos, gc", "tipo": "und"},
        {"n": "2", "titulo": "Ambiente", "sub": "Data Prep", "libs": "transformers\ntorch", "tipo": "prep"},
        {"n": "3", "titulo": "Carga", "sub": "Data Prep", "libs": "sqlite3\npandas", "tipo": "prep"},
        {"n": "4", "titulo": "Limpeza", "sub": "Data Prep", "libs": "re\nstring", "tipo": "prep"},
        {"n": "5", "titulo": "Tópicos", "sub": "Modeling", "libs": "bertopic\nsklearn, nltk", "tipo": "mod"},
        {"n": "6", "titulo": "Sentimento", "sub": "Modeling", "libs": "pysentimiento\ntqdm", "tipo": "mod"},
        {"n": "7", "titulo": "Avaliação", "sub": "Evaluation", "libs": "matplotlib\nseaborn, wordcloud", "tipo": "eval"}
    ]

    # Esquema de cores refinado para diferenciar Data Prep
    cores = {
        "und":  {"face": "#F5F5F5", "edge": "#9E9E9E", "text": "#424242", "lib_color": "#616161"}, # Cinza
        "prep": {"face": "#FFF3E0", "edge": "#FF9800", "text": "#E65100", "lib_color": "#EF6C00"}, # Laranja (Data Prep)
        "mod":  {"face": "#E3F2FD", "edge": "#1976D2", "text": "#0D47A1", "lib_color": "#1565C0"}, # Azul
        "eval": {"face": "#E8F5E9", "edge": "#388E3C", "text": "#1B5E20", "lib_color": "#2E7D32"}  # Verde
    }

    n_etapas = len(etapas)
    box_w, box_h = 1.25, 0.95
    espacamento = 1.65

    # Linha conectora de fundo
    ax.plot([0, (n_etapas-1) * espacamento], [0.5, 0.5], color='#F0F0F0',
            linewidth=15, zorder=1, solid_capstyle='round')

    for i, etapa in enumerate(etapas):
        x = i * espacamento
        y = 0.5
        estilo = cores[etapa["tipo"]]

        # 1. Box da Etapa
        rect = patches.FancyBboxPatch(
            (x - box_w/2, y - box_h/2), box_w, box_h,
            boxstyle="round,pad=0.04", linewidth=2.8,
            edgecolor=estilo["edge"], facecolor=estilo["face"], zorder=3
        )
        ax.add_patch(rect)

        # 2. Rótulo Etapa X
        ax.text(x - box_w/2, y + box_h/2 + 0.08, f"Etapa {etapa['n']}",
                fontsize=13, fontweight='bold', color='#757575', ha='left')

        # 3. Título Principal (Max)
        ax.text(x, y + 0.25, etapa["titulo"], ha='center', va='center',
                fontsize=18, fontweight='black', color=estilo["text"], zorder=4)

        # 4. Subtítulo CRISP-DM
        ax.text(x, y + 0.08, etapa["sub"], ha='center', va='center',
                fontsize=12, style='italic', color=estilo["text"], alpha=0.9, zorder=4)

        # 5. Bibliotecas Monospace
        ax.text(x, y - 0.22, etapa["libs"], ha='center', va='center',
                fontsize=12, fontweight='bold', color=estilo["lib_color"],
                family='monospace', zorder=4)

        # 6. Setas (Cores seguem a origem do fluxo)
        if i < n_etapas - 1:
            ax.annotate("", xy=(x + espacamento - box_w/2 - 0.06, y),
                        xytext=(x + box_w/2 + 0.06, y),
                        arrowprops=dict(arrowstyle='-|>', color=estilo["edge"],
                        lw=2.5, mutation_scale=25), zorder=2)

    ax.set_xlim(-1.0, (n_etapas - 1) * espacamento + 1.0)
    ax.set_ylim(-0.1, 1.1)
    ax.axis('off')

    plt.tight_layout()

    file_name = "workflow_crisp_data_prep_final.jpg"
    plt.savefig(os.path.join(EXPORT_PATH, file_name), dpi=300, bbox_inches='tight', pad_inches=0.1)

    print(f"✅ Workflow com Data Prep destacado gerado: {file_name}")
    plt.show()

gerar_workflow_data_prep_destaque()